# WFS 1.1.0 vs. WFS 1.0.0 — was sich unterscheidet

Die WFS-Dienste der Stadt Zürich werden im OGD-Katalog mit **`VERSION=1.1.0`** publiziert.
Früher stand dort `1.0.0`, und der Server beherrscht beide Versionen weiterhin.

Dieses Notebook vergleicht die beiden Versionen **am selben Dienst**, damit klar wird, was man
beim Umstieg anpassen muss — und was nicht.

**Das Ergebnis vorweg:**

| | 1.0.0 vs. 1.1.0 |
|---|---|
| **Capabilities-Dokument** | unterschiedlich aufgebaut — hier liegt die ganze Arbeit |
| **GeoJSON-Datenantwort** | **byte-identisch** |
| **Parameter** (`bbox`, `Filter`, `EXP_FILTER`, `startIndex`, `maxFeatures`, `resultType`) | identisches Verhalten |
| **GML-Standardformat** | 1.0.0 → GML 2.1.2, 1.1.0 → GML 3.1.1 |

Wer nur GeoJSON bezieht, muss beim Versionswechsel also **nichts** ändern ausser der
Versionsnummer. Wer die Capabilities auswertet, muss seinen Parser anpassen.

> Die eigentliche Anleitung zum Arbeiten mit dem WFS steht im Notebook
> [Geoportal-Beispiele.ipynb](Geoportal-Beispiele.ipynb). Dieses hier ist eine Ergänzung
> für alten Code und für alle, die beide Versionen bedienen müssen.

In [1]:
%pip install --quiet requests

Note: you may need to restart the kernel to use updated packages.


In [2]:
import xml.etree.ElementTree as ET

import requests

wfs_url = "https://www.ogd.stadt-zuerich.ch/wfs/geoportal/Statistische_Quartiere"
layer = "adm_statistische_quartiere_map"
VERSIONEN = ("1.0.0", "1.1.0")

namespaces = {
    'wfs': 'http://www.opengis.net/wfs',
    'ows': 'http://www.opengis.net/ows',
}


def get_feature(version, **extra):
    """GetFeature-Request in der angegebenen WFS-Version."""
    return requests.get(wfs_url, params={
        'service': 'WFS',
        'version': version,
        'request': 'GetFeature',
        'typename': layer,
        **extra,
    })


def localname(element):
    """Elementname ohne Namespace -- ET.QName hat KEIN .localname."""
    return element.tag.split('}')[-1]

## 1. Das Capabilities-Dokument

Schon die oberste Ebene ist anders gegliedert. In 1.0.0 beschreibt ein `Service`-Block den Dienst,
in 1.1.0 übernehmen das die OWS-Blöcke `ServiceIdentification`, `ServiceProvider` und
`OperationsMetadata`.

In [3]:
capabilities = {}
for version in VERSIONEN:
    r = requests.get(wfs_url, params={
        'service': 'WFS',
        'version': version,
        'request': 'GetCapabilities',
    })
    r.raise_for_status()
    capabilities[version] = ET.fromstring(r.content)

    print(f"--- angefragt: {version}  ({len(r.content)} Bytes)")
    print(f"    gelieferte Version: {capabilities[version].get('version')}")
    print(f"    Kindelemente      : {[localname(c) for c in capabilities[version]]}")
    print()

--- angefragt: 1.0.0  (3409 Bytes)
    gelieferte Version: 1.0.0
    Kindelemente      : ['Service', 'Capability', 'FeatureTypeList', 'Filter_Capabilities']

--- angefragt: 1.1.0  (6125 Bytes)
    gelieferte Version: 1.1.0
    Kindelemente      : ['ServiceIdentification', 'ServiceProvider', 'OperationsMetadata', 'FeatureTypeList', 'Filter_Capabilities']



## 2. Die Layer-Beschreibung — die häufigste Fehlerquelle

Beide Versionen haben eine `FeatureTypeList` mit `FeatureType`-Einträgen. Aber die Kindelemente
heissen anders:

In [4]:
for version in VERSIONEN:
    feature_types = capabilities[version].findall(
        'wfs:FeatureTypeList/wfs:FeatureType', namespaces)
    print(f"{version}: {[localname(c) for c in feature_types[0]]}")

1.0.0: ['Name', 'Title', 'SRS', 'LatLongBoundingBox', 'Operations']
1.1.0: ['Name', 'Title', 'DefaultSRS', 'OtherSRS', 'OtherSRS', 'Operations', 'WGS84BoundingBox']


Daraus folgt die Gegenüberstellung:

| Information | WFS 1.0.0 | WFS 1.1.0 |
|---|---|---|
| Koordinatensystem | `<SRS>` | `<DefaultSRS>` + `<OtherSRS>` |
| Ausdehnung | `<LatLongBoundingBox>` | `<ows:WGS84BoundingBox>` |

**Das ist die Ursache des klassischen Fehlers.** Alter Code, der auf `wfs:SRS` zugreift, bekommt
in 1.1.0 ein `None` zurück:

In [5]:
feature_type = capabilities['1.1.0'].find(
    'wfs:FeatureTypeList/wfs:FeatureType', namespaces)

try:
    srs = feature_type.find('wfs:SRS', namespaces).text
except AttributeError as e:
    print("AttributeError:", e)
    print()
    print("-> In 1.1.0 gibt es kein <SRS>-Element, find() liefert None,")
    print("   und None hat kein Attribut .text.")

AttributeError: 'NoneType' object has no attribute 'text'

-> In 1.1.0 gibt es kein <SRS>-Element, find() liefert None,
   und None hat kein Attribut .text.


Zwei Dinge helfen dagegen:

1. **`findtext()` statt `find().text`** — es gibt einen Default zurück statt `None.text` zu werfen.
2. **Beide Elementnamen abfragen**, wenn der Code mit beiden Versionen umgehen soll.

In [6]:
def read_layers(root):
    """Liest die Layer aus 1.0.0- oder 1.1.0-Capabilities."""
    result = {}
    for feature_type in root.findall('wfs:FeatureTypeList/wfs:FeatureType', namespaces):
        name = feature_type.findtext('wfs:Name', namespaces=namespaces)
        result[name] = {
            # 1.1.0: DefaultSRS -- 1.0.0: SRS
            'default_srs': (feature_type.findtext('wfs:DefaultSRS', namespaces=namespaces)
                            or feature_type.findtext('wfs:SRS', namespaces=namespaces)
                            or None),
            'other_srs': [e.text for e in feature_type.findall('wfs:OtherSRS', namespaces)],
        }
    return result


for version in VERSIONEN:
    print(version, "->", read_layers(capabilities[version]))
    print()

1.0.0 -> {'adm_statistische_quartiere_b_p': {'default_srs': None, 'other_srs': []}, 'adm_statistische_quartiere_map': {'default_srs': None, 'other_srs': []}, 'adm_statistische_quartiere_v': {'default_srs': None, 'other_srs': []}}

1.1.0 -> {'adm_statistische_quartiere_b_p': {'default_srs': None, 'other_srs': ['EPSG:4326', 'EPSG:3857']}, 'adm_statistische_quartiere_map': {'default_srs': None, 'other_srs': ['EPSG:4326', 'EPSG:3857']}, 'adm_statistische_quartiere_v': {'default_srs': None, 'other_srs': ['EPSG:4326', 'EPSG:3857']}}



Beide Versionen melden dieselben drei Layer, und in beiden ist `default_srs` **leer** — dieser
Server füllt weder `<SRS>` noch `<DefaultSRS>` aus. Das CRS aus den Capabilities zu lesen
funktioniert hier also grundsätzlich nicht, unabhängig von der Version. Man legt es stattdessen
beim Abruf selbst fest (`srsName=EPSG:2056`), oder man verlässt sich auf den GeoJSON-Standard
EPSG:4326.

Ein Unterschied zeigt sich trotzdem: **1.1.0 nennt über `OtherSRS` immerhin die unterstützten
Koordinatensysteme** (`EPSG:4326`, `EPSG:3857`), 1.0.0 kennt dieses Element gar nicht und schweigt
dazu. Das ist das einzige Stück Information, das die neuere Version hier zusätzlich liefert.

## 3. Die Ausgabeformate stehen an einer anderen Stelle

| | WFS 1.0.0 | WFS 1.1.0 |
|---|---|---|
| Pfad | `Capability/Request/GetFeature/ResultFormat` | `ows:OperationsMetadata` → `ows:Operation[@name='GetFeature']` → `ows:Parameter[@name='outputFormat']` |
| Schreibweise | Elementnamen: `<GML2/>`, `<GeoJSON/>` | MIME-Types als Werte: `application/vnd.geo+json` |

In [7]:
def read_output_formats(root):
    """Liest die GetFeature-Ausgabeformate aus 1.1.0- oder 1.0.0-Capabilities."""
    # WFS 1.1.0
    for operation in root.findall('ows:OperationsMetadata/ows:Operation', namespaces):
        if operation.get('name') != 'GetFeature':
            continue
        for parameter in operation.findall('ows:Parameter', namespaces):
            if parameter.get('name') == 'outputFormat':
                return [v.text for v in parameter.findall('ows:Value', namespaces)]
    # WFS 1.0.0
    result_format = root.find(
        'wfs:Capability/wfs:Request/wfs:GetFeature/wfs:ResultFormat', namespaces)
    if result_format is not None:
        return [localname(child) for child in result_format]
    return []


for version in VERSIONEN:
    print(f"{version}: {read_output_formats(capabilities[version])}")

1.0.0: ['GML2', 'GML3', 'GeoJSON']
1.1.0: ['text/xml; subtype=gml/2.1.2', 'text/xml; subtype=gml/3.1.1', 'application/vnd.geo+json']


Die Formate sind dieselben, nur anders benannt: `GeoJSON` ≙ `application/vnd.geo+json`,
`GML2` ≙ `text/xml; subtype=gml/2.1.2`, `GML3` ≙ `text/xml; subtype=gml/3.1.1`.

Praktisch: Der Kurzname **`outputFormat=GeoJSON` funktioniert in beiden Versionen**, obwohl ihn
1.1.0 so gar nicht mehr deklariert.

## 4. Die Datenantwort ist identisch

Das ist die gute Nachricht. Derselbe GetFeature-Request in beiden Versionen liefert dasselbe
GeoJSON — nicht nur inhaltlich gleich, sondern Byte für Byte:

In [8]:
antworten = {}
for version in VERSIONEN:
    r = get_feature(version, outputFormat='GeoJSON')
    r.raise_for_status()
    antworten[version] = r.content
    daten = r.json()
    print(f"{version}: {len(r.content):>7} Bytes, {len(daten['features'])} Features, "
          f"erste Koordinate {daten['features'][0]['geometry']['coordinates'][0][0]}")

print()
print("Antworten byte-identisch:", antworten['1.0.0'] == antworten['1.1.0'])

1.0.0:  905006 Bytes, 34 Features, erste Koordinate [8.556701, 47.364065]


1.1.0:  905006 Bytes, 34 Features, erste Koordinate [8.556701, 47.364065]

Antworten byte-identisch: True


Auch mit `srsName=EPSG:2056` sind beide Antworten gleich — insbesondere ist die
**Achsenreihenfolge dieselbe** (Ostwert, Nordwert).

Das ist erwähnenswert, weil WFS 1.1.0 im Standard eigentlich die Achsenreihenfolge des
EPSG-Registers vorschreibt, was bei manchen Servern zu vertauschten Koordinaten führt. Hier
passiert das nicht, solange man die Kurzform `EPSG:2056` verwendet:

In [9]:
for version in VERSIONEN:
    r = get_feature(version, outputFormat='GeoJSON', srsName='EPSG:2056', maxFeatures=1)
    r.raise_for_status()
    daten = r.json()
    print(f"{version}: crs={daten['crs']['properties']['name']}, "
          f"erste Koordinate {daten['features'][0]['geometry']['coordinates'][0][0]}")

1.0.0: crs=urn:ogc:def:crs:EPSG::2056, erste Koordinate [2684457.0343, 1246514.804396]
1.1.0: crs=urn:ogc:def:crs:EPSG::2056, erste Koordinate [2684457.0343, 1246514.804396]


## 5. Unterschied beim GML-Standardformat

Ohne `outputFormat` liefert jede Version ihren eigenen GML-Dialekt:

In [10]:
for version in VERSIONEN:
    r = get_feature(version, maxFeatures=1)
    r.raise_for_status()
    print(f"{version}: {r.headers['content-type']}")

1.0.0: text/xml; subtype=gml/2.1.2; charset=utf-8


1.1.0: text/xml; subtype=gml/3.1.1; charset=utf-8


Und die beiden GML-Dialekte beschreiben Geometrien unterschiedlich:

| | GML 2.1.2 (Standard in 1.0.0) | GML 3.1.1 (Standard in 1.1.0) |
|---|---|---|
| Feature-ID | `fid="..."` | `gml:id="..."` |
| Ausdehnung | `<gml:Box>` mit `<gml:coordinates>` | `<gml:Envelope>` mit `<gml:lowerCorner>` / `<gml:upperCorner>` |
| Polygon-Ring | `<outerBoundaryIs>` → `<LinearRing>` | `<exterior>` → `<Ring>` → `<curveMember>` → `<LineString>` |
| Koordinatenliste | `<coordinates>`, Paare mit Komma | `<posList>`, alle Werte mit Leerzeichen |

In [11]:
for version in VERSIONEN:
    r = get_feature(version, maxFeatures=1)
    r.raise_for_status()
    start = r.text.find('<gml:featureMember')
    print("=" * 70)
    print(version, "-", r.headers['content-type'])
    print("=" * 70)
    print(r.text[start:start + 620])
    print()

1.0.0 - text/xml; subtype=gml/2.1.2; charset=utf-8
<gml:featureMember>
 <qgs:adm_statistische_quartiere_map fid="adm_statistische_quartiere_map.1">
  <gml:boundedBy>
   <gml:Box>
    <gml:coordinates cs="," ts=" ">2684454.0423,1245344.463896 2687039.4349,1247714.156196</gml:coordinates>
   </gml:Box>
  </gml:boundedBy>
  <qgs:geometry>
   <Polygon xmlns="http://www.opengis.net/gml">
    <outerBoundaryIs xmlns="http://www.opengis.net/gml">
     <LinearRing xmlns="http://www.opengis.net/gml">
      <coordinates xmlns="http://www.opengis.net/gml" cs="," ts=" ">2684457.0343,1246514.804396 2684466.3153,1246518.996296 2684479.1642,1246524.798996 2684493.2052,1246531.13



1.1.0 - text/xml; subtype=gml/3.1.1; charset=utf-8
<gml:featureMember>
 <qgs:adm_statistische_quartiere_map gml:id="adm_statistische_quartiere_map.1">
  <gml:boundedBy>
   <gml:Envelope>
    <gml:lowerCorner>2684454.0423 1245344.463896</gml:lowerCorner>
    <gml:upperCorner>2687039.4349 1247714.156196</gml:upperCorner>
   </gml:Envelope>
  </gml:boundedBy>
  <qgs:geometry>
   <Polygon xmlns="http://www.opengis.net/gml">
    <exterior xmlns="http://www.opengis.net/gml">
     <Ring xmlns="http://www.opengis.net/gml">
      <curveMember xmlns="http://www.opengis.net/gml">
       <LineString xmlns="http://www.opengis.net/gml">
        <posList xmlns="http://www.openg



Beide Dialekte lassen sich aber in **beiden** Versionen anfordern — `outputFormat` entscheidet,
nicht die Version:

In [12]:
for version in VERSIONEN:
    for fmt in ('text/xml; subtype=gml/2.1.2', 'text/xml; subtype=gml/3.1.1'):
        r = get_feature(version, outputFormat=fmt, maxFeatures=1)
        angefordert = fmt.split('=')[-1]
        # Content-Type ist z.B. "text/xml; subtype=gml/3.1.1; charset=utf-8"
        geliefert = [teil.split('=')[-1] for teil in r.headers['content-type'].split(';')
                     if 'subtype' in teil][0]
        print(f"  WFS {version}, angefordert {angefordert}  ->  geliefert {geliefert}")

  WFS 1.0.0, angefordert gml/2.1.2  ->  geliefert gml/2.1.2


  WFS 1.0.0, angefordert gml/3.1.1  ->  geliefert gml/3.1.1


  WFS 1.1.0, angefordert gml/2.1.2  ->  geliefert gml/2.1.2
  WFS 1.1.0, angefordert gml/3.1.1  ->  geliefert gml/3.1.1


## 6. Was sich *nicht* unterscheidet

Alle gebräuchlichen Parameter verhalten sich in beiden Versionen gleich. Die folgende Zelle prüft
das der Reihe nach:

In [13]:
tests = {
    'maxFeatures=3': {'maxFeatures': 3},
    'startIndex=5 & maxFeatures=1': {'startIndex': 5, 'maxFeatures': 1},
    'count=3 (WFS 2.0.0)': {'count': 3},
    'Filter ohne xmlns': {'Filter': '<Filter><PropertyIsEqualTo>'
                                    '<PropertyName>knr</PropertyName><Literal>7</Literal>'
                                    '</PropertyIsEqualTo></Filter>'},
    'Filter MIT xmlns': {'Filter': '<Filter xmlns="http://www.opengis.net/ogc">'
                                   '<PropertyIsEqualTo>'
                                   '<PropertyName>knr</PropertyName><Literal>7</Literal>'
                                   '</PropertyIsEqualTo></Filter>'},
    'EXP_FILTER': {'EXP_FILTER': 'knr = 7'},
    'bbox in LV95': {'bbox': '2676000,1245000,2690000,1255000,EPSG:2056'},
}

print(f"{'Parameter':32s} {'WFS 1.0.0':>22s}   {'WFS 1.1.0':>22s}")
print("-" * 80)
for name, extra in tests.items():
    spalten = []
    for version in VERSIONEN:
        r = get_feature(version, outputFormat='GeoJSON', **extra)
        if 'json' in r.headers.get('content-type', ''):
            spalten.append(f"{len(r.json()['features'])} Features")
        else:
            spalten.append(f"HTTP {r.status_code} (kein JSON)")
    print(f"{name:32s} {spalten[0]:>22s}   {spalten[1]:>22s}")

Parameter                                     WFS 1.0.0                WFS 1.1.0
--------------------------------------------------------------------------------


maxFeatures=3                                3 Features               3 Features

startIndex=5 & maxFeatures=1                 1 Features               1 Features


count=3 (WFS 2.0.0)                         34 Features              34 Features
Filter ohne xmlns                            4 Features               4 Features


Filter MIT xmlns                   HTTP 403 (kein JSON)     HTTP 403 (kein JSON)
EXP_FILTER                                   4 Features               4 Features


bbox in LV95                                33 Features              33 Features


Drei Beobachtungen dazu:

* **`count` wirkt in keiner der beiden Versionen** — der Parameter stammt aus WFS 2.0.0. Statt
  einen Fehler zu melden, liefert der Server stillschweigend *alle* Features. In 1.0.0 und 1.1.0
  heisst der Parameter `maxFeatures`.
* **Der 403 bei `Filter MIT xmlns` trifft beide Versionen gleichermassen.** Das ist kein
  WFS-Thema, sondern die Web Application Firewall vor dem Dienst: eine `xmlns`-Deklaration in
  einem GET-Parameter wird blockiert. Abhilfe: `xmlns` weglassen oder POST verwenden.
* `resultType=hits` liefert in beiden Versionen das Attribut `numberOfFeatures` (erst WFS 2.0.0
  benennt es in `numberMatched` um):

In [14]:
for version in VERSIONEN:
    r = get_feature(version, resultType='hits')
    r.raise_for_status()
    root = ET.fromstring(r.content)
    print(f"{version}: numberOfFeatures={root.get('numberOfFeatures')} "
          f"| numberMatched={root.get('numberMatched')}")

1.0.0: numberOfFeatures=34 | numberMatched=None
1.1.0: numberOfFeatures=34 | numberMatched=None


## 7. Fazit: Was beim Umstieg zu tun ist

**Beziehen Sie nur Daten (GeoJSON)?** Dann ändern Sie die Versionsnummer — fertig. Die Antworten
sind identisch.

**Werten Sie die Capabilities aus?** Dann passen Sie diese Stellen an:

| Was | Alt (1.0.0) | Neu (1.1.0) |
|---|---|---|
| Koordinatensystem des Layers | `wfs:SRS` | `wfs:DefaultSRS` + `wfs:OtherSRS` |
| Ausdehnung des Layers | `wfs:LatLongBoundingBox` | `ows:WGS84BoundingBox` |
| Ausgabeformate | `wfs:Capability/.../wfs:ResultFormat` | `ows:OperationsMetadata/...` |
| Dienstbeschreibung | `wfs:Service` | `ows:ServiceIdentification`, `ows:ServiceProvider` |

**Verarbeiten Sie GML?** Dann geben Sie `outputFormat` explizit an, statt sich auf den Standard
zu verlassen — sonst bekommen Sie nach dem Versionswechsel plötzlich GML 3.1.1 statt GML 2.1.2.

**Unabhängig von der Version** gilt bei diesem Dienst:

* Das Koordinatensystem steht nicht in den Capabilities — GeoJSON ist EPSG:4326, LV95 holt man
  mit `srsName=EPSG:2056`.
* `maxFeatures`, nicht `count`.
* Keine `xmlns`-Deklaration in einem GET-`Filter`.

Zwei Hinweise zum Python-Code, die in beiden Welten gelten:

* `find('...').text` wirft `AttributeError`, sobald das Element fehlt — `findtext('...')` ist
  robuster.
* `ET.QName(element)` hat **kein** Attribut `.localname`. Den Namen ohne Namespace bekommt man
  mit `element.tag.split('}')[-1]`.

## Weiterführende Links

* [Geoportal-Beispiele.ipynb](Geoportal-Beispiele.ipynb) — die Anleitung für WFS 1.1.0
* [README dieses Verzeichnisses](README.md)
* [OGD-Katalog: Statistische Quartiere](https://data.stadt-zuerich.ch/dataset/geo_statistische_quartiere)
* [OGC-Spezifikation WFS 1.1.0](https://www.ogc.org/publications/standard/wfs/)